## Introduction

This notebook builds a Vision Transformer (ViT) from scratch in PyTorch and develops the intuition behind each component as the model takes shape. It follows the complete path from an input image to classification logits: dividing the image into patch tokens, preserving spatial information with positional embeddings, exchanging information through multi-head self-attention, refining representations with Transformer encoder blocks, and training the assembled model. Along the way, the explanations connect ViTs to familiar convolutional-network ideas and examine practical design choices such as patch size, embedding width, attention heads, depth, and dropout.

In a Vision Transformer (ViT), **patch embedding** and **positional encoding** provide two different kinds of information:

- Patch embedding answers: **"What visual content is in this patch?"**
- Positional encoding answers: **"Where did this patch come from in the image?"**

#### Patch Embedding

An image is divided into fixed-size patches. For example, a $224 \times 224$ RGB image with $16 \times 16$ patches produces:

$$
N = \frac{224}{16} \times \frac{224}{16} = 14 \times 14 = 196
$$

patches.

Each patch initially has:

$$
16 \times 16 \times 3 = 768
$$

pixel values. A learned linear projection maps each flattened patch to a model embedding of dimension $D$:

$$
\mathbf{e}_i = \mathbf{r}_i \mathbf{W}_E + \mathbf{b}_E
$$

where:

- $\mathbf{r}_i$ is the flattened raw-pixel vector for patch $i$
- $\mathbf{W}_E$ is a learned projection matrix
- $\mathbf{e}_i \in \mathbb{R}^{D}$ is the patch embedding

In PyTorch, this is often implemented efficiently with a convolution:

```python
patch_embedding = nn.Conv2d(
    in_channels=3,
    out_channels=embedding_dim,
    kernel_size=patch_size,
    stride=patch_size,
)
```

The convolution extracts non-overlapping patches and projects each one into the transformer's embedding space.

In [22]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        x = self.proj(x)  # (batch, embed_dim, height / patch_size, width / patch_size)
        x = x.flatten(2)  # (batch, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (batch, num_patches, embed_dim)
        return x

# Test the patch embedding module on two random input images
patch_embedding = PatchEmbedding()
emb = patch_embedding(torch.randn(2, 3, 1024, 1024))
print(emb)
print(emb.shape)

tensor([[[-0.2662,  1.1254,  0.1963,  ..., -0.2777,  0.3962,  0.0265],
         [ 0.0148,  0.3595,  0.0728,  ...,  0.4341,  0.2921,  1.1499],
         [ 0.7755, -0.7754,  0.8336,  ..., -0.0577, -0.1771, -0.2064],
         ...,
         [ 0.1184, -0.4542,  0.3191,  ..., -0.4793,  0.0417, -0.8668],
         [-0.2405,  0.3833,  0.1904,  ..., -0.0775, -0.3256, -0.0352],
         [ 0.6430,  0.7997,  0.5273,  ..., -0.5304,  0.2011, -0.8569]],

        [[ 1.0174,  0.1471, -0.4266,  ..., -0.8798, -0.6824, -0.4291],
         [ 0.2693, -0.4648,  0.0241,  ..., -0.3631, -0.5594,  0.0886],
         [ 0.3445,  1.0503, -0.6525,  ..., -0.1107,  0.3187, -0.1639],
         ...,
         [-0.5591, -0.1260, -0.3024,  ..., -0.2180, -0.6211, -0.0042],
         [-1.2531,  0.3697, -0.1166,  ..., -0.4816,  1.2556,  1.5264],
         [-0.7632,  0.7177, -0.4077,  ...,  0.1172, -0.1576,  0.9372]]],
       grad_fn=<TransposeBackward0>)
torch.Size([2, 4096, 768])


#### Positional Encoding

Self-attention does not inherently understand token order or spatial position. Without positional information, it treats the patch embeddings like an unordered set.

A positional vector is therefore added to every patch embedding. We denote the resulting token for patch $i$ as:

$$
\mathbf{x}_i^{(0)} = \mathbf{e}_i + \mathbf{p}_i
$$

where:

- $\mathbf{e}_i$ describes the visual content of patch $i$
- $\mathbf{p}_i$ describes the patch's position
- $\mathbf{x}_i^{(0)}$ contains both content and position

Collecting all patch tokens gives the initial Transformer sequence:

$$
X^{(0)} = E + P
$$

This $X^{(0)}$ is the same token sequence passed into the first multi-head self-attention block. More generally, $X^{(\ell)}$ denotes the sequence entering encoder block $\ell$, and that block produces $X^{(\ell+1)}$. Later sections may write $X$ when referring to the current token sequence without naming a particular layer.

In the Python implementation, this sequence is represented by the variable `x`. The mathematical symbols use uppercase $X$ for the complete token matrix and lowercase $\mathbf{x}_i$ for one token.

For learned positional embeddings:

```python
position_embedding = nn.Parameter(
    torch.randn(1, num_patches + 1, embedding_dim)
)
```

The extra position is usually for the `[CLS]` token.

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_len+1, embed_dim))

    def forward(self, x):
        x = x + self.pos_embedding[:, :x.size(1), :]
        return x

position_encoding = PositionalEncoding(embed_dim=768)
output = position_encoding.forward(torch.randn(2, 196, 768))
print(output)
print(output.shape)

tensor([[[ 0.0293, -0.1964,  0.3883,  ..., -0.3639, -2.0639,  1.3130],
         [ 0.5872, -0.6621,  0.9056,  ..., -0.2303, -0.2008, -0.3447],
         [-0.2912,  0.1886,  0.9864,  ..., -1.2744, -0.7912,  0.4923],
         ...,
         [-0.9221, -1.1735,  1.4046,  ..., -1.1884, -0.7861,  1.2260],
         [ 0.3343,  0.4235, -1.4888,  ...,  0.4311, -2.0389, -0.5199],
         [-0.3345,  0.0171,  0.6617,  ..., -1.6152, -0.8239, -0.3477]],

        [[-0.1730, -0.1917,  0.2440,  ...,  1.0526,  0.1115,  1.4806],
         [ 1.2659, -0.3111,  0.3192,  ...,  0.5134, -1.1315,  0.7498],
         [ 0.6946, -1.0140, -1.7278,  ..., -1.3158, -0.3372, -1.5579],
         ...,
         [-0.9044,  0.4978, -1.0838,  ..., -0.0046, -0.0766,  0.6288],
         [-0.9640, -0.4236,  0.5410,  ...,  0.9219,  1.4269,  0.1143],
         [ 0.4684, -0.0323,  1.3354,  ..., -0.3306, -0.7844,  0.1483]]],
       grad_fn=<AddBackward0>)
torch.Size([2, 196, 768])


#### Intuitive Comparison

A useful intuition is:

- **Patch embedding is a feature extractor.** The convolution divides the image into patches and converts the pixels in each patch into a feature vector. It tells the transformer **what is in each patch**.
- **Positional encoding is a spatial address or coordinate label.** It adds a different learned vector to each patch token to tell the transformer **where that patch came from** in the image.

For example, the patch embedding may detect features resembling an eye and a mouth. Positional encoding lets the model distinguish an eye above a mouth from a mouth above an eye. It is like attaching row-and-column coordinates to every patch before giving the patches to the transformer.

The two vectors have the same embedding dimension but different jobs:

| Component | Intuitive role | Information supplied |
|---|---|---|
| Patch embedding | Convolutional feature extraction | What appears in the patch |
| Positional encoding | Spatial address / coordinate label | Where the patch belongs |

Both are necessary because self-attention can compare patch content globally, but by itself it has no built-in knowledge of token order or the image's 2D layout. Without patch embeddings there are no visual features to analyze; without positional encodings the transformer sees those features as an unordered collection of patches.

## Patch Embedding vs. a Full Convolutional Network Stack

Patch embedding does **not always work better** than a full convolutional feature extractor with pooling. It is mainly better suited to the **Transformer architecture and its token-based input format**.

### What Patch Embedding Does

A ViT patch embedding is itself a convolution:

```python
nn.Conv2d(
    in_channels=3,
    out_channels=embed_dim,
    kernel_size=patch_size,
    stride=patch_size,
  )
```

With `kernel_size=16` and `stride=16`, it:

1. Splits the image into non-overlapping $16 \times 16$ patches.
2. Projects each patch into an embedding vector.
3. Produces a sequence of tokens for the Transformer.

For a $224 \times 224$ image:

$$
224 \times 224 \longrightarrow 14 \times 14 \longrightarrow 196\text{ tokens}
$$

Its purpose is primarily **tokenization and projection**, not sophisticated feature extraction.

### Full Convolutional Network

A conventional CNN applies several operations:

```text
Convolution -> Activation -> Convolution -> Pooling -> ...
```

This builds a hierarchy:

- Early layers detect edges and textures.
- Middle layers detect shapes and parts.
- Deeper layers detect objects and semantic concepts.

Pooling gradually reduces spatial resolution while preserving important local features.

### Why ViTs Use Simple Patch Embedding

#### 1. Transformers need tokens

A Transformer expects a sequence shaped approximately like:

$$
(\text{batch}, \text{number of tokens}, \text{embedding dimension})
$$

Patch embedding converts the image directly into that format:

$$
(B,C,H,W) \rightarrow (B,N,D)
$$

A full CNN could also produce tokens, but it would perform much of the feature extraction before the Transformer receives the image.

#### 2. Self-attention performs the later feature extraction

In a CNN, stacked convolutions extract increasingly complex features. In a ViT, the Transformer blocks perform this job through self-attention and MLP layers.

The intended division of work is:

```text
Patch embedding -> create initial visual tokens
Transformer blocks -> learn relationships and higher-level features
```

A deep CNN before the Transformer may therefore duplicate some of the Transformer's role.

#### 3. Patch embedding preserves more raw information

Pooling is intentionally lossy. Max pooling retains only the maximum value from each local region:

$$
\begin{bmatrix}
1 & 3 \\
2 & 7
\end{bmatrix}
\longrightarrow 7
$$

Other values are discarded. This gives CNNs useful translation invariance, but exact spatial details can disappear.

Patch embedding learns a projection of all values in each patch rather than explicitly selecting only the maximum. However, patch embedding is still lossy when `embed_dim` is smaller than the flattened patch dimension.

#### 4. It allows global interactions early

CNNs initially combine information only from nearby pixels. Their receptive field expands gradually as layers are stacked.

After patch embedding, self-attention allows every patch to interact directly with every other patch:

$$
\operatorname{Attention}(Q,K,V) =
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{D}}\right)V
$$

A patch in the top-left corner can immediately attend to one in the bottom-right corner. This is useful when recognizing an object depends on relationships between distant regions.

#### 5. It imposes fewer assumptions

Convolutions assume that:

- Nearby pixels are especially related.
- The same local detector should be used everywhere.
- Local features should be composed hierarchically.

These are valuable **inductive biases**, especially when training data is limited.

A ViT imposes fewer of these assumptions and allows attention to learn relationships from data. With sufficiently large datasets and models, this flexibility can lead to better performance.

### When CNNs Can Be Better

CNNs often perform better when:

- The training dataset is small.
- Compute or memory is limited.
- Local textures are especially important.
- Strong translation invariance is useful.
- Efficient inference is required.

Self-attention has approximately quadratic complexity in the number of tokens:

$$
O(N^2)
$$

Reducing the patch size increases $N$ quickly. For a $224 \times 224$ image:

- $16 \times 16$ patches produce $196$ tokens.
- $8 \times 8$ patches produce $784$ tokens.
- $4 \times 4$ patches produce $3136$ tokens.

### Hybrid Approaches

Many modern vision models combine both ideas:

```text
Convolutional stem
        |
        v
Visual feature map
        |
        v
Flatten into tokens
        |
        v
Transformer blocks
```

A shallow convolutional stem can extract stable local features and reduce resolution before attention handles global relationships.

| Approach | Strength |
|---|---|
| Patch embedding | Simple tokenization, early global attention, and scalability with large datasets |
| Deep CNN with pooling | Strong local feature extraction and good data efficiency |
| CNN-Transformer hybrid | Combines local inductive bias with global attention |

The key point is that patch embedding is not inherently a better feature extractor than a full CNN. It is a **simpler interface between images and Transformers**, allowing the Transformer itself to learn most of the feature hierarchy.

## Multi-Head Attention in a Vision Transformer

Multi-head attention and activation functions solve **different problems**:

- **Multi-head attention mixes information between image patches.**
- **An activation function transforms features nonlinearly at each location.**

Multi-head attention is therefore not the Transformer equivalent of ReLU or GELU.

### What Self-Attention Does

After patch embedding and positional encoding, the image is represented as a sequence of patch tokens:

$$
X \in \mathbb{R}^{N \times D}
$$

where $N$ is the number of patch tokens and $D$ is the embedding dimension. Self-attention allows every patch to collect information from every other patch:

$$
\operatorname{Attention}(Q,K,V) =
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d}}\right)V
$$

For each patch token:

- **Query ($Q$):** What information am I looking for?
- **Key ($K$):** What information do I contain?
- **Value ($V$):** What information should I provide if selected?

This lets the model learn relationships such as an eye attending to another eye, a wheel attending to a vehicle body, or an object attending to its surrounding context. A `[CLS]` token, when used, can attend to all patches and collect information for image classification.

Unlike an ordinary convolution, attention is **content-dependent**: the importance assigned to another patch changes with the current image.

### Why Use Multiple Heads?

A single attention operation learns one attention pattern. Multiple heads let the model examine several types of relationships in parallel. For example, different heads may learn to emphasize:

- Nearby patches and local textures
- Distant but visually similar patches
- Relationships between object parts
- Foreground and background context

Each head has its own learned query, key, and value projections:

$$
Q_h = XW_h^Q, \qquad K_h = XW_h^K, \qquad V_h = XW_h^V
$$

The head outputs are concatenated and projected back to the model dimension:

$$
\operatorname{MHA}(X) =
\operatorname{Concat}(\text{head}_1, \ldots, \text{head}_H)W^O
$$

These roles are not assigned manually. Each head learns whichever patterns help minimize the training loss, and some heads may learn overlapping or redundant patterns.

### Comparison with a CNN Activation Function

An activation function such as ReLU operates element by element:

$$
\operatorname{ReLU}(x) = \max(0,x)
$$

Its main purpose is to introduce **nonlinearity**. Without activation functions, several convolutional layers would collapse into one equivalent linear transformation:

$$
W_3(W_2(W_1x)) = W_{\text{combined}}x
$$

An activation function does not compare image locations, move information between patches, or enlarge the receptive field. Multi-head attention does all three by dynamically routing information between tokens.

| Operation | Main purpose | Mixes locations? | Input-dependent routing? |
|---|---|---:|---:|
| Convolution | Extract and combine local features | Locally | No |
| Pooling | Reduce spatial resolution | Locally | Limited |
| ReLU/GELU | Introduce nonlinearity | No | No |
| Self-attention | Exchange information between tokens | Globally | Yes |
| Multi-head attention | Learn several interaction patterns | Globally | Yes |

A closer CNN analogy for attention is a **dynamic, image-dependent convolution with a global receptive field**, not an activation function.

A convolution applies learned weights that remain fixed after training:

$$
y_i = \sum_{j \in \mathcal{N}(i)} w_j x_{i+j}
$$

Attention computes weights from the current input:

$$
y_i = \sum_{j=1}^{N} a_{ij}(X)v_j
$$

Therefore, $a_{ij}(X)$ can change from one image to another.

### ViTs Still Need Activation Functions

A typical ViT block contains both multi-head attention and an MLP with a GELU activation:

```text
Patch tokens
    |
Layer normalization
    |
Multi-head attention       <- communication between patches
    |
Residual connection
    |
Layer normalization
    |
Linear -> GELU -> Linear   <- nonlinear feature transformation
    |
Residual connection
```

Conceptually:

- **Attention asks:** Which other patches should this patch obtain information from?
- **Multiple heads ask:** Which different relationships should be examined?
- **GELU asks:** How should the resulting features be transformed nonlinearly?

Multi-head attention provides spatial communication and a global receptive field. GELU inside the ViT's MLP is the actual counterpart to activation functions such as ReLU in a convolutional network.

## Why Attention Uses Separate Queries, Keys, and Values

The input $X$ is projected into queries, keys, and values because attention must learn two distinct things:

1. **Which tokens should communicate?** Queries and keys determine the attention weights.
2. **What information should be communicated?** Values contain the information that is aggregated.

For one attention head:

$$
Q = XW^Q, \qquad K = XW^K, \qquad V = XW^V
$$

The attention weights and output are then computed as:

$$
A = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)
$$

$$
\operatorname{head}(X) = AV
$$

The outputs of completed heads are concatenated only **after** each head has performed attention:

$$
\operatorname{MHA}(X) =
\operatorname{Concat}(\operatorname{head}_1, \ldots, \operatorname{head}_H)W^O
$$

Conceptually, for a patch token:

- **Query:** What information am I looking for?
- **Key:** What searchable characteristics do I contain?
- **Value:** What information should I send if another token selects me?

This resembles information retrieval: the query is a search request, the key is a searchable label, and the value is the retrieved content.

### Why Not Use $X$ Directly?

A simplified attention operation could use:

$$
A = \operatorname{softmax}\left(\frac{XX^\top}{\sqrt{D}}\right),
\qquad Y = AX
$$

This is mathematically valid, but the same representation must then perform all three jobs: express what a token needs, describe how it should be matched, and hold the information it sends.

Separate projections let the model match patches using one set of features while transmitting another. For example, queries and keys might match patches by shape while values carry color or texture information.

Separate query and key projections also allow directional relationships. With $Q=XW^Q$ and $K=XW^K$, the fact that token $i$ needs information from token $j$ does not require token $j$ to need the same information from token $i$.

### Can One Linear Layer Compute Q, K, and V?

Yes. Efficient implementations commonly use one fused linear operation:

```python
qkv = nn.Linear(embed_dim, 3 * embed_dim)
query, key, value = qkv(x).chunk(3, dim=-1)
```

This is one linear-layer call, but its weight matrix contains three independently learned parameter blocks:

$$
W^{QKV} = \begin{bmatrix} W^Q & W^K & W^V \end{bmatrix}
$$

Using one fused operation is computationally efficient. Using one **shared representation** for $Q$, $K$, and $V$ would be more restrictive because matching features and transmitted features could not specialize independently.

Each attention head has its own projections:

$$
Q_h=XW_h^Q, \qquad K_h=XW_h^K, \qquad V_h=XW_h^V
$$

This lets different heads learn different compatibility rules and transmit different types of information. In short: **queries and keys determine who communicates, values determine what is communicated, and multiple heads provide several learned communication channels.**

## Intuition for Query, Key, and Value

An intuitive way to understand $Q$, $K$, and $V$ is as a **learned lookup-and-routing system**:

- **Query:** What information does this patch need?
- **Key:** What kind of information does this patch offer?
- **Value:** What information should this patch send if selected?

They exist simultaneously because every token acts as both a requester of information and a candidate source of information.

### Visual Example

Suppose one image patch contains part of a wheel. Its query might represent:

**"I need information about other vehicle parts."**

Other patches expose keys such as:

```text
Patch containing road      -> key: road-like
Patch containing wheel     -> key: circular vehicle part
Patch containing car body  -> key: large vehicle structure
Patch containing sky       -> key: background
```

The wheel query is compared with all those keys. Strong query-key matches receive large attention weights. The selected patches then send their values:

```text
Car-body key   -> helps decide whether to attend
Car-body value -> provides the visual features to retrieve
```

The key determines whether information is relevant; the value contains the information that is transferred.

### Key-Value Database Analogy

Attention resembles querying a key-value database:

```text
Query: "Find information relevant to this wheel"
Key:   "This patch represents a vehicle body"
Value: Learned visual features from the vehicle-body patch
```

The process is:

```text
Query matches keys
        |
Match scores become attention weights
        |
Weights select and combine values
```

Mathematically:

$$
a_{ij} = \operatorname{softmax}_j\left(
\frac{q_i k_j^\top}{\sqrt{d_k}}
\right)
$$

$$
y_i = \sum_j a_{ij}v_j
$$

Token $i$ uses its query $q_i$ to search the keys $k_j$, then receives a weighted combination of the corresponding values $v_j$.

### Traditional Deep-Learning Analogy

The closest CNN intuition combines:

- Parallel $1 \times 1$ convolution-like projections
- A learned gating mechanism
- Dynamic spatial feature aggregation

You can imagine three learned branches:

```text
Input feature map or token sequence
|-- Query branch -> features used to request information
|-- Key branch   -> features used to advertise information
|-- Value branch -> features that will be transmitted
```

The query and key branches produce a **dynamic routing matrix**. That routing matrix is applied to the value branch:

```text
Q and K -> dynamic spatial weights
V       -> features being mixed
```

This resembles gating in recurrent networks, dynamic convolution, squeeze-and-excitation, and non-local neural-network blocks. A **non-local block** is the closest traditional vision comparison because it also computes relationships between distant spatial positions and aggregates their features.

### Comparison with Ordinary Convolution

A convolution computes:

$$
y_i = \sum_{j \in \mathcal{N}(i)} w_j x_{i+j}
$$

Its weights $w_j$ are learned during training and then fixed for every image. Attention computes:

$$
y_i = \sum_j a_{ij}(X)v_j
$$

Its routing weights $a_{ij}(X)$ are recomputed for every input image.

Therefore:

- A CNN kernel asks: **Which local pattern does this fixed filter detect?**
- Attention asks: **For this token and this image, which other tokens are currently useful?**

### Why Three Representations?

A patch might be selected based on **shape** but transmit information about **color and texture**:

```text
Query and key features -> shape-based matching
Value features         -> color and texture content
```

Separate learned branches allow matching and communication to specialize:

```python
query = query_projection(x)
key = key_projection(x)
value = value_projection(x)
```

The branches receive the same input but learn different views because they use different weights.

All three representations are needed at the same time because attention must evaluate candidate relationships before producing an output:

1. Every token produces a query.
2. Every token produces a key.
3. Every token produces a value.
4. All queries are compared with all keys.
5. The resulting weights combine the values.

For self-attention:

$$
Q=XW^Q, \qquad K=XW^K, \qquad V=XW^V
$$

These are three learned views of the same tokens, not three copies with identical meaning.

The compact intuition is: **$Q$ is the request, $K$ is the label used for matching, and $V$ is the payload. Attention uses the requests and labels to decide how to route the payloads.**

## Why Use Dropout in Multi-Head Attention?

The `dropout` argument in PyTorch's `nn.MultiheadAttention` regularizes the **attention probabilities**. After computing the attention scores, the module forms:

$$
A = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)
$$

During training, dropout randomly sets some entries of $A$ to zero before the values are aggregated:

$$
\widetilde{A} = \operatorname{Dropout}(A),
\qquad \operatorname{Attention}(Q,K,V)=\widetilde{A}V
$$

In the encoder block:

```python
self.mha = nn.MultiheadAttention(
    embed_dim,
    num_heads,
    dropout=dropout,
    batch_first=True,
  )
```

a value such as `dropout=0.1` means that attention connections are randomly dropped with probability $0.1$ during training.

### Why Regularize Attention Connections?

Without attention dropout, a head may become too dependent on a small number of token-to-token relationships. For example, one patch might repeatedly place almost all its attention on one other patch. That relationship may work well for the training images but generalize poorly.

Attention dropout encourages the model to:

- Avoid relying on one specific patch relationship
- Use alternative contextual evidence
- Distribute learning across more token connections and attention heads
- Reduce overfitting, especially on smaller datasets

An intuitive analogy is temporarily hiding some communication links between patches during training. The model must still form a useful representation using the remaining links.

Attention dropout does **not** permanently remove patches or attention heads. A dropped attention link can be active again on the next training pass.

### Why Is There Also Dropout in the MLP?

Attention dropout and MLP dropout regularize different computations:

| Dropout location | What is randomly suppressed | Purpose |
|---|---|---|
| MHA dropout | Attention probabilities between token pairs | Prevent over-reliance on particular token relationships |
| MLP dropout | Hidden or projected feature activations | Prevent over-reliance on particular feature channels |

The encoder uses both because attention communicates **between tokens**, while the MLP transforms features **within each token**. Regularizing one operation does not fully regularize the other.

### Training and Evaluation Behavior

Dropout is active only in training mode:

```python
block.train()  # dropout is active
```

It is disabled in evaluation mode:

```python
block.eval()   # dropout is disabled
```

When dropout is active, repeated forward passes can produce different outputs. In evaluation mode, the output is deterministic for the same input and model parameters.

### How Much Dropout Should Be Used?

Typical starting values are:

- `0.0` for very large datasets, strong augmentation, or models that do not overfit
- `0.1` as a common default
- `0.2` or higher only when validation results show substantial overfitting

Too much dropout can remove useful attention links and cause underfitting. The best value should be selected using validation performance rather than assuming that more regularization is better.

The concise intuition is: **attention dropout regularizes who communicates, while MLP dropout regularizes which feature activations are used.**

In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim, 
            num_heads,
            batch_first=True)

    def forward(self, x):
        return self.attn(x, x, x)

# Test the MultiHeadAttention module
embed_dim = 64
num_heads = 8
seq_length = 10
batch_size = 2

x = torch.rand(batch_size, seq_length, embed_dim)
mha = MultiHeadAttention(embed_dim, num_heads)
output, _ = mha(x)
print(output)
print(output.shape)

tensor([[[-0.3228, -0.1382, -0.0740,  ..., -0.1819, -0.2628, -0.1078],
         [-0.3292, -0.1403, -0.0726,  ..., -0.1824, -0.2596, -0.1067],
         [-0.3297, -0.1416, -0.0729,  ..., -0.1833, -0.2593, -0.1106],
         ...,
         [-0.3221, -0.1369, -0.0748,  ..., -0.1856, -0.2613, -0.1097],
         [-0.3226, -0.1402, -0.0736,  ..., -0.1830, -0.2626, -0.1058],
         [-0.3250, -0.1387, -0.0733,  ..., -0.1859, -0.2646, -0.1066]],

        [[-0.2574, -0.0502, -0.1336,  ..., -0.1740, -0.2637, -0.1036],
         [-0.2560, -0.0456, -0.1394,  ..., -0.1754, -0.2645, -0.1041],
         [-0.2595, -0.0484, -0.1327,  ..., -0.1788, -0.2603, -0.1023],
         ...,
         [-0.2565, -0.0467, -0.1352,  ..., -0.1753, -0.2632, -0.1022],
         [-0.2584, -0.0480, -0.1350,  ..., -0.1745, -0.2648, -0.1022],
         [-0.2596, -0.0466, -0.1366,  ..., -0.1781, -0.2651, -0.1025]]],
       grad_fn=<TransposeBackward0>)
torch.Size([2, 10, 64])


## Choosing the Number of Attention Heads

There is no universally optimal number of attention heads. It is a **hyperparameter** chosen jointly with the embedding dimension, model size, dataset, and compute budget.

### Head Dimension

If the model embedding dimension is $D$ and there are $H$ heads, each head typically has dimension:

$$
d_{\text{head}} = \frac{D}{H}
$$

Therefore, $D$ normally needs to be divisible by $H$. For example, with $D=768$:

| Heads $H$ | Head dimension $d_{\text{head}}$ |
|---:|---:|
| 6 | 128 |
| 8 | 96 |
| 12 | 64 |
| 16 | 48 |
| 24 | 32 |

The number of heads creates a tradeoff:

- **Too few heads:** The model has fewer independent attention patterns.
- **Too many heads:** Each head becomes narrow and may lack enough capacity to represent useful relationships.
- **More heads do not automatically improve accuracy:** Heads can become redundant or contribute little.

A common starting target is:

$$
d_{\text{head}} \approx 32\text{ to }64
$$

This is a useful heuristic, not a mathematical optimum.

### Practical Starting Points

| Embedding dimension | Heads | Head dimension |
|---:|---:|---:|
| 192 | 3 or 6 | 64 or 32 |
| 384 | 6 | 64 |
| 512 | 8 | 64 |
| 768 | 12 | 64 |
| 1024 | 16 | 64 |

The original ViT configurations commonly follow this pattern:

- ViT-Base: $D=768$, $H=12$
- ViT-Large: $D=1024$, $H=16$
- ViT-Huge: $D=1280$, $H=16$, giving $d_{\text{head}}=80$

The example above uses:

```python
embed_dim = 64
num_heads = 8
```

Therefore:

$$
d_{\text{head}} = \frac{64}{8} = 8
$$

This is valid and useful for demonstrating the mechanics, although a head dimension of $8$ may be narrow for a practical image model. For a small real model with $D=64$, starting with `num_heads=2` or `4` gives head dimensions of $32$ or $16$.

### How to Find the Best Value

Treat the number of heads as a validation hyperparameter:

1. Hold the embedding dimension and training setup constant.
2. Try a small set of divisors of $D$.
3. Compare validation accuracy, loss, memory use, and latency.
4. Choose the smallest configuration that achieves the desired result.

For $D=256$, for example:

```text
num_heads in {4, 8}
head_dim  in {64, 32}
```

Testing every possible divisor is usually unnecessary.

At fixed $D$, increasing $H$ does not greatly change the parameter count of the standard query, key, value, and output projections. Their combined parameter scale remains roughly:

$$
4D^2
$$

However, more heads can increase implementation overhead and memory usage, even when the main arithmetic cost remains similar.

## Are Attention Heads Like Multiple CNN Kernels?

It is a **partially useful analogy**, but not an exact one.

A CNN layer has many kernels:

```text
Kernel 1 -> may detect vertical edges
Kernel 2 -> may detect horizontal edges
Kernel 3 -> may detect textures
```

Multiple attention heads may similarly learn different patterns:

```text
Head 1 -> nearby patch relationships
Head 2 -> repeated textures
Head 3 -> distant object parts
Head 4 -> foreground-background relationships
```

In that limited sense, both create multiple parallel feature-processing pathways. The major difference is how their weights are used.

### CNN Kernel

A convolution kernel contains fixed learned weights after training:

$$
y_i = \sum_{j \in \mathcal{N}(i)} w_j x_{i+j}
$$

The kernel applies the same local pattern at every image position. Its receptive field is usually local.

### Attention Head

An attention head calculates interaction weights from the current input:

$$
a_{ij}(X) =
\operatorname{softmax}_j\left(
\frac{q_i k_j^\top}{\sqrt{d_{\text{head}}}}
\right)
$$

Its output is:

$$
y_i = \sum_{j=1}^{N} a_{ij}(X)v_j
$$

The weights $a_{ij}(X)$:

- Depend on the current image
- Can differ for every query patch
- Can connect distant patches
- Change when the input changes

A better intuition is:

**A CNN kernel is a fixed local feature detector, while an attention head is a learned feature subspace combined with a dynamic, input-dependent routing pattern.**

CNN kernels mainly detect **what local pattern exists**. Attention heads mainly determine **which tokens should exchange information**, although their value projections also transform features.

### Summary

- Choose $H$ so that $D$ is divisible by $H$.
- Start with a head dimension around $32$ to $64$.
- Compare a few configurations using validation performance.
- Do not assume that more heads are better.
- Multiple heads resemble multiple CNN kernels only because both provide parallel learned pathways.
- Unlike fixed local CNN kernels, attention heads dynamically connect tokens based on the current input.

## Choosing the Embedding Dimension

There is no universally optimal embedding dimension. The embedding dimension $D$ controls how much information each patch token can represent and is one of the main determinants of a ViT's capacity, memory use, and computational cost.

### What the Embedding Dimension Means

After patch embedding, each image patch is represented by a vector:

$$
\mathbf{x}_i \in \mathbb{R}^{D}
$$

A larger $D$ gives the model more channels for representing visual information such as texture, color, shape, object identity, and context.

For a $16 \times 16$ RGB patch, the raw patch contains:

$$
16 \times 16 \times 3 = 768
$$

values. However, the embedding dimension does **not** have to be $768$. The patch projection can compress or expand that vector:

$$
\mathbb{R}^{768} \rightarrow \mathbb{R}^{D}
$$

For example, $D$ could be `192`, `384`, or `768`.

### Capacity and Compute Tradeoff

#### Smaller Embedding Dimension

Advantages:

- Faster training and inference
- Lower GPU memory usage
- Lower overfitting risk on small datasets
- Suitable for prototypes and edge devices

Disadvantages:

- Less representational capacity
- May create an information bottleneck
- Can limit performance on complex datasets

#### Larger Embedding Dimension

Advantages:

- Greater feature capacity
- Supports more or wider attention heads
- Often improves performance when sufficient data is available

Disadvantages:

- More parameters
- Greater memory usage
- Higher computational cost
- Greater overfitting risk on limited data

Many Transformer operations scale approximately as:

$$
O(ND^2)
$$

where $N$ is the number of tokens. Doubling $D$ can therefore make projection and MLP computations roughly four times as expensive.

Self-attention also contains the token-interaction cost:

$$
O(N^2D)
$$

Thus, both the token count and embedding dimension matter.

### Practical Starting Points

| Use case | Suggested $D$ | Typical heads |
|---|---:|---:|
| Educational example | 64-128 | 2-4 |
| Small custom dataset | 128-256 | 4-8 |
| Lightweight ViT | 192-384 | 3-6 |
| Medium-scale model | 512-768 | 8-12 |
| Large pretrained model | 768-1280+ | 12-16+ |

Common published configurations include:

| Model | Embedding dimension | Heads | Head dimension |
|---|---:|---:|---:|
| ViT-Tiny | 192 | 3 | 64 |
| ViT-Small | 384 | 6 | 64 |
| ViT-Base | 768 | 12 | 64 |
| ViT-Large | 1024 | 16 | 64 |
| ViT-Huge | 1280 | 16 | 80 |

These are established configurations, not universal optima.

### Relationship to the Number of Heads

Choose $D$ and the number of heads $H$ together:

$$
d_{\text{head}} = \frac{D}{H}
$$

Useful requirements and heuristics:

- $D$ must normally be divisible by $H$.
- A head dimension around $32$ to $64$ is a common starting point.
- Avoid selecting so many heads that each head becomes extremely narrow.

Examples:

```python
embed_dim = 256
num_heads = 4  # head_dim = 64
```

```python
embed_dim = 384
num_heads = 6  # head_dim = 64
```

```python
embed_dim = 768
num_heads = 12  # head_dim = 64
```

### Choosing It Experimentally

For a custom model, start with the smallest plausible configuration and scale only when validation results justify it:

1. Select two or three candidate dimensions, such as `192`, `384`, and `768`.
2. Keep the patch size, number of layers, training schedule, and dataset split fixed.
3. Select compatible head counts.
4. Compare validation accuracy, loss, memory consumption, and inference latency.
5. Increase $D$ if both training and validation performance indicate underfitting.
6. Reduce $D$ or add regularization if training improves while validation performance deteriorates.

A useful experiment could be:

| Experiment | $D$ | Heads | Head dimension |
|---|---:|---:|---:|
| Small | 192 | 3 | 64 |
| Medium | 384 | 6 | 64 |
| Large | 768 | 12 | 64 |

### Recommendation for This Notebook

The current demonstration uses:

```python
embed_dim = 64
num_heads = 8
```

This gives:

$$
d_{\text{head}} = \frac{64}{8} = 8
$$

It works as an educational example, but eight dimensions per head is relatively narrow. A clearer small example would be:

```python
embed_dim = 64
num_heads = 2  # head_dim = 32
```

For a small trainable ViT, a reasonable starting configuration is:

```python
embed_dim = 192
num_heads = 3  # head_dim = 64
```

The practical rule is:

**Use the smallest embedding dimension that provides sufficient validation performance, and choose the head count so each head retains enough capacity. Larger dimensions are useful only when the dataset, task complexity, and compute budget can support them.**

## Transformer Encoder Block

A Transformer encoder block refines every patch token using two complementary operations:

1. **Multi-head self-attention** exchanges information between tokens.
2. **The MLP** transforms each token's features independently.

The block preserves the input shape:

$$
X \in \mathbb{R}^{B \times N \times D}
\quad \longrightarrow \quad
Y \in \mathbb{R}^{B \times N \times D}
$$

where $B$ is the batch size, $N$ is the number of tokens, and $D$ is the embedding dimension. Preserving $D$ allows residual additions and makes it possible to stack many encoder blocks.

### 1. Pre-Normalization

The implementation first normalizes each token's feature vector:

```python
normalized_x = self.norm1(x)
```

Layer normalization stabilizes feature scales and improves optimization. Because normalization happens before attention, this is a **pre-norm Transformer**:

$$
\hat{X} = \operatorname{LayerNorm}(X)
$$

Pre-norm blocks are generally easier to train when many encoder layers are stacked.

### 2. Multi-Head Self-Attention

The normalized tokens are passed as query, key, and value:

```python
attention_output, _ = self.mha(
    normalized_x,
    normalized_x,
    normalized_x,
    need_weights=False,
  )
```

Because all three inputs come from the same token sequence, this is **self-attention**. Each patch can gather context from every other patch. Setting `need_weights=False` avoids returning an attention matrix that this block does not use.

The attention output has the same shape as $X$:

$$
\operatorname{MHA}(\hat{X}) \in \mathbb{R}^{B \times N \times D}
$$

### 3. First Residual Connection

The attention result is added to the original input:

```python
x = x + attention_output
```

Mathematically:

$$
X' = X + \operatorname{MHA}(\operatorname{LayerNorm}(X))
$$

The residual path preserves the original token representation while attention contributes contextual information. It also gives gradients a direct route through deep networks.

### 4. Token-Wise MLP

The second normalized representation passes through an MLP:

```python
self.mlp = nn.Sequential(
    nn.Linear(embed_dim, mlp_dim),
    nn.GELU(),
    nn.Dropout(dropout),
    nn.Linear(mlp_dim, embed_dim),
    nn.Dropout(dropout),
)
```

The first linear layer expands each token from $D$ to `mlp_dim`; GELU introduces nonlinearity; the second linear layer projects it back to $D$. The same MLP is applied independently to every token: it transforms features but does not mix token positions.

Typically, `mlp_dim` is larger than `embed_dim`, often by a factor near four:

$$
D \rightarrow D_{\text{MLP}} \rightarrow D,
\qquad D_{\text{MLP}} \approx 4D
$$

Attention communicates **between tokens**, while the MLP computes **within each token's feature vector**.

### 5. Second Residual Connection

The MLP result is added back to its input:

```python
x = x + self.mlp(self.norm2(x))
```

Therefore:

$$
Y = X' + \operatorname{MLP}(\operatorname{LayerNorm}(X'))
$$

The complete pre-norm block is:

$$
X' = X + \operatorname{MHA}(\operatorname{LN}(X))
$$

$$
Y = X' + \operatorname{MLP}(\operatorname{LN}(X'))
$$

### Role of Each Component

| Component | Purpose |
|---|---|
| Layer normalization | Stabilizes the feature distribution and training |
| Multi-head attention | Exchanges information globally between patch tokens |
| First residual connection | Preserves the original tokens and supports gradient flow |
| MLP with GELU | Performs nonlinear feature transformation within each token |
| Dropout | Regularizes attention and MLP transformations during training |
| Second residual connection | Preserves contextualized tokens and supports deep stacking |

### Why This Definition Is Appropriate

The implementation below is a valid modern ViT encoder block because it:

- Uses `batch_first=True`, matching patch tensors shaped `(batch, tokens, embed_dim)`.
- Uses pre-normalization before both sublayers.
- Uses multi-head self-attention for token communication.
- Uses GELU in the MLP, which is conventional for Transformers.
- Uses dropout for regularization.
- Uses residual connections around both attention and the MLP.
- Returns the same shape it receives.

For the example configuration:

```python
embed_dim = 64
num_heads = 8
mlp_dim = 128
```

the tensor shape remains `(2, 10, 64)` throughout the block, while attention mixes information across the 10 tokens and the MLP transforms each token's 64 features.

### Why Is `normalized_x` Passed Three Times?

In this call:

```python
attention_output, _ = self.mha(
    normalized_x,  # query source
    normalized_x,  # key source
    normalized_x,  # value source
    need_weights=False,
  )
```

the three arguments specify the source tensors for the **query**, **key**, and **value** roles. Passing the same tensor for all three tells `nn.MultiheadAttention` to perform **self-attention**.

The supplied tensors are not yet $Q$, $K$, and $V$. The module internally applies separately learned projections:

$$
Q = XW^Q, \qquad K = XW^K, \qquad V = XW^V
$$

Conceptually, this is similar to:

```python
query = normalized_x @ W_q
key = normalized_x @ W_k
value = normalized_x @ W_v
```

PyTorch may calculate these projections with one fused operation for efficiency, but they still use distinct learned parameter blocks.

Therefore, `normalized_x` is passed three times because all three roles originate from the same sequence, **not because parallel computation requires three copies**. Python passes references to the tensor and does not copy its underlying data three times.

The three-argument interface also allows the same module to perform cross-attention, where the query comes from one sequence and the keys and values come from another.

In [23]:
class TransformerEncoderBlock(nn.Module):
    def __init__(
        self,
        embed_dim,
        num_heads,
        mlp_dim,
        dropout=0.1,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        normalized_x = self.norm1(x)
        attention_output, _ = self.mha(
            normalized_x,
            normalized_x,
            normalized_x,
            need_weights=False,
        )
        x = x + attention_output
        x = x + self.mlp(self.norm2(x))
        return x


# Test the TransformerEncoderBlock module
embed_dim = 64
num_heads = 8
mlp_dim = 128
seq_length = 10
batch_size = 2

x = torch.rand(batch_size, seq_length, embed_dim)
block = TransformerEncoderBlock(embed_dim, num_heads, mlp_dim)
output = block(x)
print(output)
print(output.shape)

tensor([[[ 0.4677,  0.1087,  0.6389,  ...,  0.4388,  0.7465,  0.9009],
         [-0.1318,  0.6435,  0.1901,  ...,  0.0937,  0.1054,  0.9316],
         [ 0.5428,  0.1547,  0.8977,  ...,  0.6156,  0.5335,  0.3062],
         ...,
         [ 1.0660,  0.5779,  0.4188,  ...,  0.1515, -0.5348,  1.1082],
         [ 0.9878,  0.7071, -0.0772,  ...,  0.2289,  0.6049,  0.2979],
         [ 0.6936,  0.3234,  0.4163,  ...,  0.8954,  0.0252,  0.2958]],

        [[ 0.1463,  0.7311,  0.9554,  ...,  0.4284,  0.3325,  0.6871],
         [ 0.7426,  1.2983,  0.9640,  ...,  1.0770,  0.2501,  0.0164],
         [ 1.1450,  0.9518,  0.8160,  ...,  0.2416,  0.2451,  0.4419],
         ...,
         [ 0.8112,  1.0295,  0.8847,  ...,  1.1760,  0.2051, -0.0317],
         [ 1.2626,  0.4244,  1.1833,  ...,  0.1350,  0.5425,  0.6495],
         [ 0.3402,  0.9898,  0.7090,  ...,  0.8216, -0.4655,  0.6004]]],
       grad_fn=<AddBackward0>)
torch.Size([2, 10, 64])


## Self-Attention vs. Cross-Attention

Both mechanisms use the same scaled dot-product attention calculation:

$$
\operatorname{Attention}(Q,K,V) =
\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

The difference is **where the query, key, and value inputs come from**.

### Self-Attention

In self-attention, $Q$, $K$, and $V$ are all projected from the same sequence $X$:

$$
Q=XW^Q, \qquad K=XW^K, \qquad V=XW^V
$$

```python
output, _ = attention(x, x, x)
```

Each token asks which other tokens in its own sequence are relevant. In a ViT encoder, every image-patch token can therefore exchange information with every other image-patch token.

```text
Image patch tokens X
    |
    +--> Queries
    +--> Keys
    +--> Values
            |
      Self-attention
```

For an input with shape $(B,N,D)$, the output also has shape $(B,N,D)$. The query length $N$ determines the number of output tokens.

### Cross-Attention

In cross-attention, queries come from one sequence $X$, while keys and values come from another sequence $C$:

$$
Q=XW^Q, \qquad K=CW^K, \qquad V=CW^V
$$

```python
output, _ = attention(
    query_tokens,
    context_tokens,
    context_tokens,
  )
```

The query sequence asks which information it should retrieve from the context sequence. For example:

- Text tokens can attend to image tokens in a multimodal model.
- Image tokens can attend to text tokens.
- Decoder tokens can attend to encoder outputs in an encoder-decoder Transformer.

```text
Query sequence X   --> Queries --+
                                  |--> Cross-attention --> output for X
Context sequence C --> Keys ------+
                   --> Values ----+
```

If query tokens have shape $(B,N_q,D)$ and context tokens have shape $(B,N_c,D)$, the attention score matrix has shape:

$$
(B,H,N_q,N_c)
$$

and the output has one result per query token:

$$
\text{output shape} = (B,N_q,D)
$$

### Direct Comparison

| Property | Self-attention | Cross-attention |
|---|---|---|
| Query source | Current sequence | Current/query sequence |
| Key source | Same sequence | Context sequence |
| Value source | Same sequence | Context sequence |
| Main purpose | Relate tokens within one sequence | Retrieve information from another sequence |
| Typical ViT use | Connect image patches | Fuse images with text or another modality |
| PyTorch call | `attention(x, x, x)` | `attention(x, context, context)` |

The `TransformerEncoderBlock` above uses self-attention because it passes `normalized_x` as all three source arguments. A standard ViT encoder generally needs only self-attention. Cross-attention becomes useful when the model must connect the image representation to a different representation, such as text, audio, or decoder queries.

The concise distinction is:

- **Self-attention:** tokens communicate within the same sequence.
- **Cross-attention:** one sequence retrieves information from another sequence.

In [25]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        embed_dim=768,
        num_heads=8,
        mlp_dim=1024,
        num_layers=6,
        num_classes=10,
        dropout=0.1,
    ):
        super().__init__()
        if img_size % patch_size != 0:
            raise ValueError("img_size must be divisible by patch_size")
        if embed_dim % num_heads != 0:
            raise ValueError("embed_dim must be divisible by num_heads")

        self.img_size = img_size
        self.patch_embedding = PatchEmbedding(
            patch_size, in_channels, embed_dim
        )
        num_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.positional_encoding = PositionalEncoding(embed_dim, num_patches)
        self.transformer_encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(
                embed_dim, num_heads, mlp_dim, dropout=dropout
            )
            for _ in range(num_layers)
        ])
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes),
        )

        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        # validate input size matches expected image size
        if x.shape[-2:] != (self.img_size, self.img_size):
            raise ValueError(
                f"Expected images of size {self.img_size}x{self.img_size}, "
                f"but received {x.shape[-2]}x{x.shape[-1]}"
            )

        # patchify the input images and add class token and positional encoding
        x = self.patch_embedding(x)
        # expand the class token to match the batch size
        cls_tokens = self.cls_token.expand(x.size(0), -1, -1)
        # concatenate the class token with the patch embeddings
        x = torch.cat((cls_tokens, x), dim=1)
        # add positional encoding to the patch embeddings with the class token
        x = self.positional_encoding(x)

        # pass the embeddings through the transformer encoder blocks
        for block in self.transformer_encoder_blocks:
            x = block(x)

        # extract the output corresponding to the class token
        cls_output = x[:, 0]
        # pass the class token output through the MLP head for classification
        return self.mlp_head(cls_output)


# Lightweight forward-pass test
model = VisionTransformer(
    img_size=32,
    patch_size=8,
    embed_dim=64,
    num_heads=4,
    mlp_dim=128,
    num_layers=2,
    num_classes=10,
    dropout=0.1,
 )
test_images = torch.randn(2, 3, 32, 32)
logits = model(test_images)
print(logits)
print(logits.shape)  # (2, 10)

tensor([[ 0.1012,  0.5422,  0.2649,  0.5775, -0.5526,  0.1498,  0.0085,  0.3257,
         -0.1804,  0.6869],
        [ 0.7495,  0.9544,  0.0911,  0.4001,  0.2600, -0.3661,  0.3951,  0.9183,
         -0.4808, -0.5708]], grad_fn=<AddmmBackward0>)
torch.Size([2, 10])


## Choosing the Number of Transformer Layers

There is no universally optimal number of encoder layers. The best depth depends on the dataset size, task complexity, embedding dimension, training recipe, compute budget, and acceptable inference latency.

In the model above, `num_layers` controls how many independent `TransformerEncoderBlock` instances are created:

```python
self.encoder_blocks = nn.ModuleList([
    TransformerEncoderBlock(embed_dim, num_heads, mlp_dim)
    for _ in range(num_layers)
])
```

Each block has its own attention and MLP parameters. The blocks do not share weights unless weight sharing is implemented explicitly.

### What Additional Layers Learn

The first self-attention layer already allows every patch token to interact with every other patch token. Therefore, adding depth is not primarily needed to make the receptive field global; it is already global after one attention operation.

Additional layers allow **iterative refinement**:

```text
Early layers   -> local appearance, edges, textures, simple patch relations
Middle layers  -> shapes, parts, spatial configurations
Later layers   -> objects, global context, task-specific semantics
```

A useful intuition is that one block performs one round of communication and feature transformation:

$$
X^{(\ell+1)} = \operatorname{EncoderBlock}_{\ell}(X^{(\ell)})
$$

Stacking $L$ blocks performs $L$ rounds:

$$
X^{(0)} \rightarrow X^{(1)} \rightarrow \cdots \rightarrow X^{(L)}
$$

A patch can reconsider which other patches matter after its representation has been updated by earlier blocks. This repeated reasoning is the main benefit of depth.

### Too Few Layers

A shallow model may:

- Underfit a complex dataset
- Learn simple patch relationships but miss higher-level structure
- Produce similar training and validation performance that are both unsatisfactory

If both training and validation loss remain high, increasing depth may help, provided model width and optimization are also adequate.

### Too Many Layers

An unnecessarily deep model may:

- Consume more memory and training time
- Increase inference latency
- Overfit a small dataset
- Become harder to optimize without suitable normalization, initialization, learning-rate schedules, and regularization
- Add layers whose representations contribute little to validation performance

Pre-normalization and residual connections make deep Transformers easier to optimize, but they do not guarantee that extra layers improve the task.

### Compute and Parameter Cost

At fixed token count $N$, embedding dimension $D$, and MLP dimension, the computation and parameters of the encoder stack grow approximately linearly with the number of layers $L$:

$$
\text{encoder compute} \propto L
$$

$$
\text{encoder parameters} \propto L
$$

Each layer still includes attention costs of approximately $O(N^2D)$ and projection/MLP costs involving $O(ND^2)$. Doubling the layer count therefore roughly doubles encoder computation and parameter storage, while training activation memory also increases unless techniques such as gradient checkpointing are used.

### Common Model Depths

Published ViT families often use configurations such as:

| Model scale | Typical depth | Typical embedding dimension |
|---|---:|---:|
| Tiny or Small ViT | 12 layers | 192-384 |
| ViT-Base | 12 layers | 768 |
| ViT-Large | 24 layers | 1024 |
| ViT-Huge | 32 layers | 1280 |

These are established large-scale pretraining configurations, not default choices for every custom dataset.

### Practical Starting Points

| Use case | Suggested starting depth |
|---|---:|
| Educational demonstration | 2-4 layers |
| Small custom dataset | 4-6 layers |
| Moderate dataset and compute | 6-12 layers |
| Large-scale pretraining | 12-32+ layers |

For this notebook, `num_layers=6` is a reasonable small-model starting point. It is deep enough to demonstrate repeated attention and MLP refinement without the cost of a standard 12-layer ViT-Base.

### How to Select the Depth

Treat depth as a validation hyperparameter:

1. Choose a small set of candidate depths, such as `4`, `6`, and `8`.
2. Keep the embedding dimension, head count, MLP dimension, patch size, optimizer, and training schedule fixed.
3. Compare training loss, validation performance, peak memory, and inference latency.
4. Increase depth when both training and validation metrics indicate underfitting.
5. Stop increasing depth when validation gains become negligible or latency and memory exceed the budget.
6. If training improves but validation deteriorates, use fewer layers, more data, stronger augmentation, weight decay, dropout, or stochastic depth.

A simple controlled experiment is:

| Experiment | Layers | Other settings |
|---|---:|---|
| Shallow | 4 | Fixed |
| Medium | 6 | Fixed |
| Deeper | 8 | Fixed |

Use the smallest depth that reaches the desired validation performance. This usually gives better latency and a lower overfitting risk than selecting the deepest model that fits in memory.

### Depth Must Be Chosen with Width

Depth cannot be optimized independently of `embed_dim` and `mlp_dim`:

- A narrow, deep model performs many refinement steps with limited features per token.
- A wide, shallow model represents richer features but performs fewer refinement steps.
- Increasing either depth or width raises capacity and compute.

When the compute budget is fixed, compare balanced configurations rather than increasing every dimension simultaneously. For example:

```text
Small:  embed_dim=192, layers=6,  heads=3
Medium: embed_dim=384, layers=8,  heads=6
Large:  embed_dim=768, layers=12, heads=12
```

The concise rule is: **start shallow, increase depth only while validation performance improves enough to justify the added memory and latency, and choose depth together with model width and dataset scale.**

### Training the Vision Transformer

In [26]:
import torch.optim as optim
from torchvision import datasets, transforms

# Define the image transformation pipeline for the dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load the dataset with the defined transformations
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Define the data loaders for training and testing
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

model = VisionTransformer()  # Replace with the actual Vision Transformer model initialization
criterion = torch.nn.CrossEntropyLoss()  # Define the loss function for classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Define the optimizer for training

# Move the model to the appropriate device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training loop (simplified example)
for epoch in range(10):  # Number of training epochs
    model.train()  # Set the model to training mode
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        # Zero / reset the parameter gradients
        optimizer.zero_grad()
        # Forward pass and loss computation
        outputs = model(images)
        # Compute the loss based on the model's output and the true labels
        loss = criterion(outputs, labels)
        # Backward pass and optimization step
        loss.backward()
        # Perform the optimization step to update the model's parameters
        optimizer.step()
        # Accumulate the running loss for monitoring training progress
        running_loss += loss.item()
    print(f"Epoch [{epoch+1}/10], Loss: {running_loss/len(train_loader):.4f}")

100%|██████████| 170M/170M [23:26<00:00, 121kB/s]  


Epoch [1/10], Loss: 2.3072
Epoch [2/10], Loss: 2.2080
Epoch [3/10], Loss: 2.2812
Epoch [4/10], Loss: 2.2683


KeyboardInterrupt: 

## Summary and Conclusion

A Vision Transformer converts an image-classification problem into a sequence-modeling problem. The image is split into non-overlapping patches, each patch is projected into an embedding, and positional information is added so the model retains the image's spatial structure. A trainable `[CLS]` token is prepended to the sequence and gradually gathers information from all image patches as the sequence passes through the encoder stack.

Each pre-normalized encoder block combines two complementary operations: multi-head self-attention communicates globally between tokens, while the GELU MLP transforms the features within each token. Residual connections preserve information and support gradient flow, layer normalization stabilizes training, and dropout helps reduce overfitting. After the final encoder block, the normalized `[CLS]` representation is mapped to class logits by the classification head.

The complete pipeline is:

```text
Image -> Patch embeddings -> [CLS] token + positional embeddings
      -> Repeated attention and MLP encoder blocks
      -> Final [CLS] representation -> Classification logits
```

The implementation also shows that ViT design choices must be considered together. Patch size controls token count and attention cost; embedding dimension controls representation capacity; the number of heads determines how that capacity is divided; depth controls the number of refinement stages; and the MLP width and dropout rate affect feature transformation and regularization. The best configuration is therefore not simply the largest one, but the smallest balanced model that achieves the required validation performance within the available data, memory, and latency budget.

With these components assembled, the notebook provides a complete educational ViT implementation that can be extended with stronger augmentation, improved optimization schedules, validation metrics, pretrained weights, or alternative attention and positional-encoding methods.